In [ ]:
!pip install kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 28.6 MB/s eta 0:00:00


In [ ]:
import kaleido
import time
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.special import softmax
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
from google.colab import drive
import os
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.stats import pearsonr
from copy import deepcopy

In [ ]:

drive.mount('/content/drive')

save_fig_path = "*******************"

Mounted at /content/drive




---



---
# One community simulations:

The goal is to examine the differences between traditional social networks (SN) and online social networks. This simulation aims to investigate whether factors such as the size of the network, reliance on continuous feedback (characteristic of online SN), asymmetry in connections, and constant updating of connection strengths influence the pace of group or community learning, the convergence to a shared norm or policy, and other behavioral outcomes.


In [ ]:
def binarize_adj_matrix(adj_matrix,HP):
    binarized_matrix = np.copy(adj_matrix)
    binarized_matrix[binarized_matrix >= HP['adj_mat_threshold']['high_t']] = 1
    binarized_matrix[binarized_matrix <= HP['adj_mat_threshold']['low_t']] = 0
    return binarized_matrix


def update_adj_matrix(adj_mat, pe, learning_rate=0.001):
    for j in range(adj_mat.shape[0]):
        for k in range(adj_mat.shape[1]):
            if j != k:
                adj_mat[j, k] = np.clip(adj_mat[j, k] + learning_rate * pe[j], 0, 1)
    return adj_mat



def create_adj_matrix_single_group(HP):
    total_agents = HP['n_agents']
    adj_matrix = np.zeros((total_agents, total_agents))
    for i in range(total_agents):
        for j in range(total_agents):
            if i != j:
                adj_matrix[i, j] = round(np.random.uniform(HP['adj_mat_group_conn']['in_group'], 1), 4)
    np.fill_diagonal(adj_matrix, 0)
    return adj_matrix

def create_reward_mat(HP, adj_mat):
    n_agents = HP['n_agents']  # Number of agents
    actions = HP['actions']  # List of actions
    rewards = HP['rewards']  # Reward structure for each action
    n_actions = len(actions)  # Number of actions
    reward_mat = np.zeros((n_agents, n_agents, n_actions))
    for a in range(n_actions):
        reward_mat[:, :, a] = adj_mat * rewards[actions[a]][1]
        # Apply r_self to the diagonal
        reward_mat[:, :, a] += np.eye(n_agents) * rewards[actions[a]][0]

    return reward_mat



In [ ]:

def interindividual_actor_critic_single_group(HP, adj_mat, reward_mat, is_online=False):

    n = HP['n_agents']
    n_rounds = int(HP['nt']/n) #number of rounds

    if HP['nt'] % n != 0:
        raise ValueError('number of trials must be divisible by number of agents')

    policy_all = np.zeros((HP['trials'],n_rounds,n,HP['n_actions']))
    V_all = np.zeros((HP['trials'],HP['nt'],n,HP['n_states']))
    advantage_all = np.zeros((HP['trials'],HP['nt'],n,HP['n_actions'], HP['n_states']))
    action_arr = np.arange(HP['n_actions'])
    for t in range(HP['trials']):
        if is_online:
          adj_mat = create_adj_matrix_single_group(HP)
          reward_mat = create_reward_mat(HP, adj_mat)
        # print(f"Initial adj matrix of trial #{t+1}: \n",adj_mat)
        V = np.zeros((HP['nt'],n,HP['n_states']))      #state value estimates
        pe = np.zeros(n)                   #prediction error (advantage estimate)
        reward = np.zeros(n)                #reward
        adv = np.zeros((HP['nt'],n,HP['n_actions'], HP['n_states']))                  #array to save advantage
        adv[0,:,:,:] = np.nan                                       #initial advantage is defined as nan, so that initial 0 values that remain until an action is chosen don't mess up averaging
        c = np.zeros((n,HP['n_actions']))         #control
        thetas = np.zeros((n,HP['n_actions']))    #no bias at first step
        policy = np.ones((n_rounds,n,HP['n_actions']))/HP['n_actions'] #initialize policy
        state_mat = np.eye(n).astype(int)   #state matrix, with 1 for the state of the acting agent (self acting), 0 for the state of the other agents (other acting)

        i=1 #start from the second time step for the advanatge calculation
        while i<HP['nt']-n:
         #iterate over all agents in each round and multiply the pe after computation by the adjacency matrix to avoid updating the value of unconnected agents
         #I need to think about the right approach for indexing the state (adding self connections in the adjaceny matrix or not?)

            agents_idx = np.arange(n)
            round = int(i/n)+1

            for j in range(n):
                  #sample action
                  a= np.random.choice(action_arr, size=1, p=policy[round-1,j,:])
                  a = a.astype(int)
                  #reward and prediction error
                  reward = np.squeeze(reward_mat[j,:,a])
                  pe = reward - V[i-1,agents_idx,state_mat[j]]

                  pe = adj_mat[j].T * pe            #0 out the pe of non-connected agents, only update the value of connected agents
                  #loss aversion - others
                  pe = pe * (1 - np.sign(pe) * HP['la'])
                  #pe of acting agent
                  pe[j] = reward[j] + HP['ratio']*np.sum(pe) - V[i-1,j,state_mat[j,j]]
                  pe[j] = pe[j] * (1 - np.sign(pe[j]) * HP['la'])
                  #update state value and advantage
                  V[i] = np.copy(V[i-1])
                  V[i,np.arange(n),state_mat[j]] = V[i-1,np.arange(n),state_mat[j]] + HP['lrs']*pe

                  #update advantage - current pe for the action that was taken, previous advantage for the other actions
                  adv[i] = np.copy(adv[i-1])
                  adv[i,agents_idx,a,state_mat[j]] = pe

                  #policy gradient update
                  c[j,a] = HP['lrp']*HP['dr']*policy[round-1,j,a]*(1-policy[round-1,j,a])
                  thetas[j,a] =thetas[j,a] + c[j,a]*pe[j]

                  #softmax policy update - acting agent
                  policy[round,j,:] = np.exp(HP['dr']*thetas[j])/np.sum(np.exp(HP['dr']*thetas[j]))
                  if is_online:
                      adj_mat = update_adj_matrix(adj_mat,pe,HP['lrm'])
                      adj_mat = binarize_adj_matrix(adj_mat,HP)
                  i += 1



        # print('done with trial ' + str(t+1) + ' out of ' + str(HP['trials']))
        # print(f"Final adj matrix of trial #{t+1}: \n",adj_mat)

        policy_all[t,:,:,:] = np.copy(policy)
        V_all[t,:,:,:] = np.copy(V)
        advantage_all[t,:,:,:,:] = np.copy(adv)

    return policy_all, V_all, advantage_all





---



## SN definitions:
Traditional Social Network:

In [ ]:
HP_traditional = {
    'trials': 1,
    'nt': 500,
    'n_actions': 3,
    'lrs': 0.1,
    'lrp': 0.1,
    'lrm': 0,
    'ratio': 0.6,
    'dr': 1,
    'n_states': 2,
    'la': 0,
    'n_agents': 10,
    'actions': ['selfish', 'coop', 'irrational'],
    'rewards': {'selfish': [5, -1], 'coop': [1, 1], 'irrational': [-1, -1]},
    'adj_mat_group_conn': {'in_group': 1, 'out_group': 0.0},
    'adj_mat_threshold': {'high_t': 0.95, 'low_t': 0.2}
}


Online Social Network:

In [ ]:
HP_online = {
    'trials': 1,
    'nt': 5000,
    'n_actions': 3,
    'lrs': 0.1,
    'lrp': 0.1,
    'lrm': 0.001,
    'ratio': 0.1,
    'dr': 1,
    'n_states': 2,
    'la': 0,
    'n_agents': 100,
    'actions': ['selfish', 'coop', 'irrational'],
    'rewards': {'selfish': [5, -1], 'coop': [1, 1], 'irrational': [-1, -1]},
    'adj_mat_group_conn': {'in_group': 0.5, 'out_group': 0},
    'adj_mat_threshold': {'high_t': 0.95, 'low_t': 0.2}
}


Simulation executions:

In [ ]:
# Run simulation for the traditional network
adj_matrix_traditional = create_adj_matrix_single_group(HP_traditional)
reward_mat_traditional = create_reward_mat(HP_traditional, adj_matrix_traditional)
policy_traditional, V_traditional, A_traditional = interindividual_actor_critic_single_group(
    HP_traditional,adj_matrix_traditional, reward_mat_traditional
)

print(policy_traditional.shape)
print(policy_traditional[:, -1, :, :])

(1, 50, 10, 3)
[[[0.08471824 0.86306083 0.05222093]
  [0.07821195 0.85929729 0.06249077]
  [0.09268582 0.84059714 0.06671704]
  [0.08793109 0.85333855 0.05873036]
  [0.07386382 0.88253299 0.04360319]
  [0.08563418 0.86816634 0.04619949]
  [0.1032632  0.84264306 0.05409374]
  [0.08196422 0.87248094 0.04555484]
  [0.12965121 0.77252112 0.09782767]
  [0.08883905 0.84416872 0.06699223]]]


In [ ]:
# Run simulation for the online network
adj_matrix_online = create_adj_matrix_single_group(HP_online)
reward_mat_online = create_reward_mat(HP_online, adj_matrix_online)
policy_online, V_online, A_online = interindividual_actor_critic_single_group(
    HP_online,adj_matrix_online, reward_mat_online,is_online=True
)


In [ ]:
def plot_average_action_selection_across_trials(policy_all, title="Average Action Selection Across Trials and Agents", save_dir=None, image_format="png", scale=2):
 # Ensure pio is imported

    # Time axis (rounds)
    x = list(range(policy_all.shape[1]))

    # Create the plot
    fig = go.Figure()

    # For each action, calculate the average across trials and agents
    for action_idx in range(policy_all.shape[3]):
        avg_action_across_trials_and_agents = policy_all[:, :, :, action_idx].mean(axis=(0, 2))
        fig.add_trace(go.Scatter(
            x=x,
            y=avg_action_across_trials_and_agents,
            mode='lines+markers',
            name=f"Action {action_idx}"
        ))

    # Customize layout
    fig.update_layout(
        title=title,
        xaxis_title="Round Number",
        yaxis_title="Average Probability",
        template="plotly_white",
        legend_title="Actions"
    )

    # Save to file if save_dir is provided
    if save_dir:
        safe_title = title.replace(" ", "_").replace(":", "").replace("/", "_")
        save_path = f"{save_dir}/{safe_title}.{image_format}"
        pio.write_image(fig, save_path, scale=scale)  # Ensure no duplicate 'format'
        print(f"Graph saved as: {save_path}")

    # Display the graph
    fig.show()


def plot_state_value_average_across_trials(V_all,n_agents, title="Average State Values Across Trials and Agents", save_dir=None, image_format="png", scale=2):
    import plotly.io as pio  # Ensure pio is imported

    n_rounds = V_all.shape[1] // n_agents

    # Average state values across trials and agents
    avg_state_values_across_trials_and_agents = V_all.mean(axis=(0, 2))[:n_rounds * n_agents:n_agents]


    # Create the plot
    fig = go.Figure()
    for state_idx in range(avg_state_values_across_trials_and_agents.shape[1]):
        fig.add_trace(go.Scatter(
            x = list(range(n_rounds)),
            y=avg_state_values_across_trials_and_agents[:, state_idx],
            mode='lines',
            name=f"State {state_idx}"
        ))

    # Customize layout
    fig.update_layout(
        title=title,
        xaxis_title="Round Number",
        yaxis_title="Average State Value",
        legend_title="States",
        template="plotly"
    )

    # Save to file if save_dir is provided
    if save_dir:
        safe_title = title.replace(" ", "_").replace(":", "").replace("/", "_")
        save_path = f"{save_dir}/{safe_title}.{image_format}"
        pio.write_image(fig, save_path, scale=scale)  # Ensure no duplicate 'format'
        print(f"Graph saved as: {save_path}")

    # Display the graph
    fig.show()




def plot_average_action_selection_with_shaded_std(policy_all, title="Average Action Selection Across Trials and Agents", save_dir=None, image_format="png", scale=2):
    # Time axis (rounds)
    x = list(range(policy_all.shape[1]))

    # Create the plot
    fig = go.Figure()

    # Specific colors for actions
    colors = ['blue', 'red', 'green']  # Define colors for Action 0, 1, 2

    # For each action, calculate the average and standard deviation across trials and agents
    for action_idx in range(min(3, policy_all.shape[3])):  # Ensure we only access the colors we have defined
        avg_action_across_trials_and_agents = policy_all[:, :, :, action_idx].mean(axis=(0, 2))
        std_action_across_trials_and_agents = policy_all[:, :, :, action_idx].std(axis=(0, 2))

        # Color for the current action
        color = colors[action_idx]  # Select color based on action index

        # Add trace for the average
        fig.add_trace(go.Scatter(
            x=x,
            y=avg_action_across_trials_and_agents,
            mode='lines',
            name=f"Action {action_idx} Mean",
            line=dict(color=color, width=2)
        ))

        # Add shaded region for the standard deviation
        fig.add_trace(go.Scatter(
            x=x+x[::-1],  # x, then x reversed
            y=list(avg_action_across_trials_and_agents + std_action_across_trials_and_agents) + list(avg_action_across_trials_and_agents - std_action_across_trials_and_agents)[::-1],
            fill='toself',
            fillcolor=color.replace(')', ',0.2)').replace('rgb', 'rgba'),  # Adjust color opacity for shading
            line=dict(color='rgba(255,255,255,0)'),
            opacity=0.2,
            showlegend=True,
            name=f"Action {action_idx} Std Dev"
        ))

    # Customize layout
    fig.update_layout(
        title=title,
        xaxis_title="Round Number",
        yaxis_title="Average Probability",
        template="plotly_white",
        legend_title="Actions and Variability"
    )

    # Save to file if save_dir is provided
    if save_dir:
        safe_title = title.replace(" ", "_").replace(":", "").replace("/", "_")
        save_path = f"{save_dir}/{safe_title}.{image_format}"
        pio.write_image(fig, save_path, scale=scale)
        print(f"Graph saved as: {save_path}")

    # Display the graph
    fig.show()



In [ ]:
# # # ציור הממוצע עבור כל הפעולות ברשת המסורתית
plot_average_action_selection_across_trials(policy_traditional, title="Average Action Selection - Traditional Network",save_dir=save_fig_path+'/one_group_general')

# ציור ממוצע ערכי המצב עבור הרשת המסורתית
plot_state_value_average_across_trials(V_traditional,HP_traditional['n_agents'], title="Average State Values Across All Agents - Traditional Network",save_dir=save_fig_path+'/one_group_general')


plot_average_action_selection_across_trials(policy_traditional, title="Average Action Selection - Traditional Network")

# ציור ממוצע ערכי המצב עבור הרשת המסורתית
plot_state_value_average_across_trials(V_traditional,HP_traditional['n_agents'], title="Average State Values Across All Agents - Traditional Network")

plot_average_action_selection_with_shaded_std(policy_traditional, title="Average Action Selection with STD - Traditional Network",save_dir=save_fig_path+'/one_group_general')




Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/one_group_general/Average_Action_Selection_with_STD_-_Traditional_Network.png


In [ ]:
# ציור הממוצע עבור כל הפעולות ברשת המקוונת
plot_average_action_selection_across_trials(policy_online,  title="Average Action Selection - Online Network",save_dir=save_fig_path+'/one_group_general')

# # ציור ממוצע ערכי המצב עבור הרשת המקוונת
plot_state_value_average_across_trials(V_online,HP_online['n_agents'], title="Average State Values Across All Agents - Online Network",save_dir=save_fig_path+'/one_group_general')


# ציור הממוצע עבור כל הפעולות ברשת המקוונת
plot_average_action_selection_across_trials(policy_online,  title="Average Action Selection - Online Network")

# ציור ממוצע ערכי המצב עבור הרשת המקוונת
plot_state_value_average_across_trials(V_online,HP_online['n_agents'], title="Average State Values Across All Agents - Online Network")

plot_average_action_selection_with_shaded_std(policy_online, title="Average Action Selection with STD - Online Network",save_dir=save_fig_path+'/one_group_general')

Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/one_group_general/Average_Action_Selection_with_STD_-_Online_Network.png


In [ ]:
def plot_parameter_correlation(param_values, param_name, HP, update_function,n_type, save_dir=None, image_format="png", scale=2):
    coop_preferences = []

    for value in param_values:
        print(value)
        HP_copy = deepcopy(HP)
        update_function(HP_copy, value)

        adj_matrix = create_adj_matrix_single_group(HP_copy)
        reward_mat = create_reward_mat(HP_copy, adj_matrix)
        policy, _, _ = interindividual_actor_critic_single_group(
            HP_copy, adj_matrix, reward_mat, is_online=True
        )

        coop_mean = np.mean(policy[:, -1, :, 1])
        coop_preferences.append(coop_mean)

    correlation = np.corrcoef(param_values, coop_preferences)[0, 1]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=param_values,
        y=coop_preferences,
        mode='markers+lines',
        name='Average Cooperative Action',
        marker=dict(size=10, color='blue'),
        line=dict(dash='solid')
    ))

    trendline = np.poly1d(np.polyfit(param_values, coop_preferences, 1))
    fig.add_trace(go.Scatter(
        x=param_values,
        y=trendline(param_values),
        mode='lines',
        name='Trendline',
        line=dict(color='red', dash='dash')
    ))

    fig.update_layout(
        title=f"Correlation Between {param_name} and Cooperative Action (ρ = {correlation:.2f}) {n_type} SN",
        xaxis_title=param_name,
        yaxis_title="Average Cooperative Action Preference",
        template="plotly_white"
    )

    if save_dir:
        safe_title = f"Correlation_{param_name.replace(' ', '_')}_{n_type}_SN"
        save_path = f"{save_dir}/{safe_title}.{image_format}"
        pio.write_image(fig, save_path, scale=scale)
        print(f"Graph saved as: {save_path}")

    fig.show()


Parater testing:

In [ ]:
ratios = [0.01,0.05,0.1,0.2,0.4,0.5,0.6,0.75,0.9,1]
connection_strengths = np.linspace(0, 1, 10)
network_sizes = [5,10,50,100,150,200]
connection_learning_rates = np.logspace(-4, -1, 10)



def update_ratio(HP, value):
    HP['ratio'] = value

def update_connection_strength(HP, value):
    HP['adj_mat_group_conn']['in_group'] = value
    HP['adj_mat_group_conn']['out_group'] = value / 2  # לדוגמה, קשרים מחוץ לקבוצה חלשים יותר


def update_network_size(HP, value):
    HP['n_agents'] = int(value)
    HP['nt'] = int(value * 20)  # התאמת מספר הצעדים בהתאם למספר הסוכנים

def update_connection_learning_rate(HP, value):
    HP['lrm'] = value


Traditional SN

In [ ]:

plot_parameter_correlation(
    param_values=connection_strengths,
    param_name="Connections strength",
    HP=HP_traditional,
    update_function=update_connection_strength,
    n_type = "Traditional",
    save_dir=save_fig_path+'/parameter_correlation'
)


plot_parameter_correlation(
    param_values=network_sizes,
    param_name="network Size",
    HP=HP_traditional,
    update_function=update_network_size,
    n_type = "Traditional",
    save_dir=save_fig_path+'/parameter_correlation'
)


plot_parameter_correlation(
    param_values=ratios,
    param_name="Ratios",
    HP=HP_traditional,
    update_function=update_ratio,
    n_type = "Traditional",
    save_dir=save_fig_path+'/parameter_correlation'
)


plot_parameter_correlation(
    param_values=connection_learning_rates,
    param_name="Dynamic",
    HP=HP_traditional,
    update_function=update_connection_learning_rate,
    n_type = "Traditional",
    save_dir=save_fig_path+'/parameter_correlation'
)







Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/parameter_correlation/Correlation_Connections_strength_Traditional_SN.png


Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/parameter_correlation/Correlation_network_Size_Traditional_SN.png


Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/parameter_correlation/Correlation_Ratios_Traditional_SN.png


Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/parameter_correlation/Correlation_Dynamic_Traditional_SN.png


Online SN

In [ ]:

plot_parameter_correlation(
    param_values=connection_strengths,
    param_name="Connections strength",
    HP=HP_online,
    update_function=update_connection_strength,
    n_type = "Online",
    save_dir=save_fig_path+'/parameter_correlation/online'
)


plot_parameter_correlation(
    param_values=network_sizes,
    param_name="network Size",
    HP=HP_online,
    update_function=update_network_size,
    n_type = "Online",
    save_dir=save_fig_path+'/parameter_correlation'
)


plot_parameter_correlation(
    param_values=ratios,
    param_name="Ratios",
    HP=HP_online,
    update_function=update_ratio,
    n_type = "Online",
    save_dir=save_fig_path+'/parameter_correlation'
)


plot_parameter_correlation(
    param_values=connection_learning_rates,
    param_name="Dynamic",
    HP=HP_online,
    update_function=update_connection_learning_rate,
    n_type = "Online",
    save_dir=save_fig_path+'/parameter_correlation/online'
)







9.999999999999999e-05
0.00021544346900318845
0.00046415888336127773
0.001
0.002154434690031882
0.004641588833612777
0.01
0.021544346900318822
0.046415888336127774
0.09999999999999999
Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/parameter_correlation/online/Correlation_Dynamic_Online_SN.png




---



---



# Two Group Setting simulations

In [ ]:
def create_reward_mat(HP, group_assignment):
    n_agents = HP['n_agents']
    n_actions = HP['n_actions']
    reward_mat = np.zeros((n_agents, n_agents, n_actions))

    for i in range(n_agents):
        for j in range(n_agents):
            if group_assignment[i] == group_assignment[j]:
                # תגמול חיובי עבור פעולה שמתאימה לקבוצה
                reward_mat[i, j, 0] = HP['rewards']['group1'][0] if group_assignment[i] == 0 else HP['rewards']['group2'][1]
                reward_mat[i, j, 1] = HP['rewards']['group2'][0] if group_assignment[i] == 1 else HP['rewards']['group1'][1]
            else:
                # תגמול שלילי עבור פעולה שלא מתאימה לקבוצה
                reward_mat[i, j, 0] = HP['rewards']['group1'][1]
                reward_mat[i, j, 1] = HP['rewards']['group2'][1]

            # תגמול פעולה ניטרלית (שווה לשני הצדדים)
            reward_mat[i, j, 2] = HP['rewards']['neutral'][0]

    np.fill_diagonal(reward_mat[:, :, 0], 0)
    np.fill_diagonal(reward_mat[:, :, 1], 0)
    np.fill_diagonal(reward_mat[:, :, 2], 0)

    return reward_mat





def create_grouped_adj_matrix(HP, traditional=False):
    total_nodes = HP['n_agents']
    adj_matrix = np.zeros((total_nodes, total_nodes))

    # Randomly shuffle nodes and split into groups
    nodes = np.arange(total_nodes)
    np.random.shuffle(nodes)
    groups = np.array_split(nodes, HP['d'])

    # Assign groups
    group_assignment = np.zeros(total_nodes, dtype=int)

    for group_idx, group in enumerate(groups):
        for i in group:
            group_assignment[i] = group_idx
            for j in group:
                if i != j:
                    # Create symmetric connections if traditional flag is True
                    connection_strength = round(np.random.uniform(HP['adj_mat_group_conn']['in_group'], 1), 4)
                    adj_matrix[i, j] = connection_strength
                    if traditional:
                        adj_matrix[j, i] = connection_strength

    for i in range(total_nodes):
        for j in range(total_nodes):
            if i != j and adj_matrix[i, j] == 0:
                # Create symmetric connections between groups if traditional flag is True
                connection_strength = round(np.random.uniform(0, HP['adj_mat_group_conn']['out_group']), 4)
                adj_matrix[i, j] = connection_strength
                if traditional:
                    adj_matrix[j, i] = connection_strength

    np.fill_diagonal(adj_matrix, 0)

    return adj_matrix, group_assignment


def create_new_adj_matrix(group_assignment, HP):
    total_nodes = HP['n_agents']
    adj_matrix = np.zeros((total_nodes, total_nodes))

    groups = [np.where(group_assignment == i)[0] for i in range(HP['d'])]

    for group in groups:
        for i in group:
            for j in group:
                if i != j:
                    adj_matrix[i, j] = round(np.random.uniform(HP['adj_mat_group_conn']['in_group'], 1),4)

    for i in range(total_nodes):
        for j in range(total_nodes):
            if i != j and adj_matrix[i, j] == 0:
                adj_matrix[i, j] = round(np.random.uniform(0, HP['adj_mat_group_conn']['out_group']),4)

    np.fill_diagonal(adj_matrix, 0)

    return adj_matrix

def update_adj_matrix(adj_mat, pe, learning_rate=0.001):
    for j in range(adj_mat.shape[0]):
        for k in range(adj_mat.shape[1]):
            if j != k:  # לא לעדכן את האלכסון
                adj_mat[j, k] = np.clip(adj_mat[j, k] + learning_rate * pe[j], 0, 1)
    return adj_mat

def binarize_adj_matrix(adj_matrix,HP):
    # Create a copy of the adjacency matrix to avoid modifying the original
    binarized_matrix = np.copy(adj_matrix)

    # Apply thresholds
    binarized_matrix[binarized_matrix >= HP['adj_mat_threshold']['high_t']] = 1
    binarized_matrix[binarized_matrix <= HP['adj_mat_threshold']['low_t']] = 0

    return binarized_matrix


In [ ]:
def interindividual_actor_critic(HP, group_assignment, reward_mat, is_online=False, thetas_init='uniform', return_values=False, V_init='default'):
    n = HP['n_agents']
    n_rounds = int(HP['nt'] / n)

    if HP['nt'] % n != 0:
        raise ValueError('number of trials must be divisible by number of agents')

    policy_all = np.zeros((HP['trials'], n_rounds, n, HP['n_actions']))
    V_all = np.zeros((HP['trials'], HP['nt'], n, HP['n_states']))

    if return_values:
        advantage_all = np.zeros((HP['trials'], HP['nt'], n, HP['n_actions'], HP['n_states']))

    action_arr = np.arange(HP['n_actions'])

    for t in range(HP['trials']):
        print(f"Trial #{t + 1} out of {HP['trials']}")
        V = np.zeros((HP['nt'], n, HP['n_states']))  # Ensure 3D regardless of return_values
        if return_values:
            adv = np.zeros((HP['nt'], n, HP['n_actions'], HP['n_states']))
            adv[0, :, :, :] = np.nan

        c = np.zeros((n, HP['n_actions']))
        thetas = np.zeros((n, HP['n_actions']))
        policy = np.ones((n_rounds, n, HP['n_actions'])) / HP['n_actions']
        state_mat = np.eye(n).astype(int)

        if V_init != 'default':
            V[0, :, :] = np.copy(V_init)

        if thetas_init != 'uniform':
            thetas = np.copy(thetas_init)
            policy[0, :, :] = np.exp(HP['dr'] * thetas) / np.sum(np.exp(HP['dr'] * thetas), axis=1, keepdims=True)

        i = 1
        adj_mat = create_new_adj_matrix(group_assignment, HP)
        while i < HP['nt'] - n:
            agents_idx = np.arange(n)
            round = int(i / n) + 1
            for j in np.arange(n):
                a = np.random.choice(action_arr, size=1, p=policy[round - 1, j, :]).astype(int)
                if a == 2:
                    reward = np.ones(n) * HP['rewards']['neutral'][0]
                else:
                    reward = np.squeeze(reward_mat[j, :, a])

                pe = reward - V[i - 1, agents_idx, state_mat[j]]
                if a == 2:
                    pe = reward - np.mean(V[i - 1, agents_idx, 0])

                pe = np.random.binomial(1, adj_mat[j]) * pe
                pe = pe * (1 - np.sign(pe) * HP['la'])
                pe[j] = reward[j] + HP['ratio'] * np.sum(pe) - V[i - 1, j, 1]
                pe[j] = pe[j] * (1 - np.sign(pe[j]) * HP['la'])
                V[i] = np.copy(V[i - 1])
                V[i, np.arange(n), state_mat[j]] = V[i - 1, np.arange(n), state_mat[j]] + HP['lrs'] * pe

                c[j, a] = HP['dr'] * policy[round - 1, j, a] * (1 - policy[round - 1, j, a])
                thetas[j, a] = thetas[j, a] + HP['lrp'] * c[j, a] * pe[j]
                policy[round, j, :] = softmax(thetas[j])

                if is_online:
                    adj_mat = update_adj_matrix(adj_mat, pe, HP['lrm'])
                    adj_mat = binarize_adj_matrix(adj_mat, HP)
                i += 1

        policy_all[t, :, :, :] = np.copy(policy)
        if return_values:
            V_all[t, :, :, :] = np.copy(V)
            advantage_all[t, :, :, :, :] = np.copy(adv)

    if return_values:
        return policy_all, V_all, advantage_all
    else:
        return policy_all, V


In [ ]:
HP_group_traditional = {
    'trials': 1,
    'd':2,
    'nt': 500,
    'n_agents': 10,
    'n_actions': 3,  # Selfish (0), Cooperative (1), Neutral (2)
    'n_states': 2,
    'lrs': 0.1,  # Learning rate for state value updates
    'lrp': 0.1,  # Learning rate for policy updates
    'dr': 1.0,   # Discount rate for policy
    'la': 0.1,   # Loss aversion factor
    'ratio': 0.6,  # Importance of other's feedback
    'rewards': {
        'group1': [3, -1],  # [reward_self, reward_others]
        'group2': [3, -1],
        'neutral': [1]      # Reward for neutral action
    },
    'adj_mat_group_conn': {
        'in_group': 1,    # Connection strength within groups
        'out_group': 0.3    # Connection strength between groups
    },
    'adj_mat_threshold': {'high_t': 0.95, 'low_t': 0.2},
    'lrm': 0  # No dynamic adjustment of connections
}


In [ ]:
HP_group_online = {
    'trials': 1,
    'd':2,
    'nt': 5000,
    'n_agents': 100,
    'n_actions': 3,  # Selfish (0), Cooperative (1), Neutral (2)
    'n_states': 2,
    'lrs': 0.1,  # Learning rate for state value updates
    'lrp': 0.1,  # Learning rate for policy updates
    'dr': 1.0,   # Discount rate for policy
    'la': 0.1,   # Loss aversion factor
    'ratio': 0.1,  # Importance of other's feedback
    'rewards': {
        'group1': [3, -1],  # [reward_self, reward_others]
        'group2': [3, -1],
        'neutral': [1]     # Reward for neutral action
    },
    'adj_mat_group_conn': {
        'in_group': 0.6,    # Connection strength within groups
        'out_group': 0.4    # Connection strength between groups
    },
    'adj_mat_threshold': {'high_t': 0.9, 'low_t': 0.1},
    'lrm': 0.001  # Dynamic adjustment of connections
}


In [ ]:

def plot_group_action_preferences(group1_action_preferences, group2_action_preferences, actions,save_dir=None, title="Group Action Preferences"):
    """
    Plot side-by-side bar charts for action preferences of two groups with unique colors for each action.

    Parameters:
    - group1_action_preferences: List of action preferences for Group 1.
    - group2_action_preferences: List of action preferences for Group 2.
    - actions: List of action labels.
    - title: Title of the plot.
    """
    colors = ['#636EFA', '#EF553B', '#00CC96']  # Define unique colors for actions

    # Create subplots
    fig = sp.make_subplots(rows=1, cols=2, subplot_titles=("Group 1", "Group 2"))

    # Add Group 1 bars
    for i, action in enumerate(actions):
        fig.add_trace(
            go.Bar(x=[action], y=[group1_action_preferences[i]], name=f"Group 1: {action}", marker_color=colors[i]),
            row=1, col=1
        )

    # Add Group 2 bars
    for i, action in enumerate(actions):
        fig.add_trace(
            go.Bar(x=[action], y=[group2_action_preferences[i]], name=f"Group 2: {action}", marker_color=colors[i]),
            row=1, col=2
        )

    # Update layout
    fig.update_layout(
        title=title,
        showlegend=False,
        template="plotly_white"
    )

    fig.update_xaxes(title_text="Actions", row=1, col=1)
    fig.update_xaxes(title_text="Actions", row=1, col=2)
    fig.update_yaxes(title_text="Average Preference", row=1, col=1)
    fig.update_yaxes(title_text="Average Preference", row=1, col=2)
    if save_dir:
        safe_title = title.replace(" ", "_").replace(":", "").replace("/", "_")
        save_path = f"{save_dir}/{safe_title}.png"
        pio.write_image(fig, save_path, scale=2)  # Ensure no duplicate 'format'
        print(f"Graph saved as: {save_path}")

    # Display the figure
    fig.show()


def plot_action_preferences_over_rounds(policy_all, group_assignment, actions,save_dir=None, title="Action Preferences Over Rounds"):
    # Colors for actions
    colors = ['#636EFA', '#EF553B', '#00CC96']  # Unique colors for actions

    # Calculate average preferences over trials
    avg_policy = policy_all.mean(axis=0)

    # Separate preferences by groups
    group1_policy = avg_policy[:, group_assignment == 0, :].mean(axis=1)
    group2_policy = avg_policy[:, group_assignment == 1, :].mean(axis=1)

    # Create a figure
    fig = sp.make_subplots(rows=1, cols=2, subplot_titles=("Group 1", "Group 2"))

    # Add lines for each action for Group 1
    for i, action in enumerate(actions):
        fig.add_trace(
            go.Scatter(x=list(range(group1_policy.shape[0])), y=group1_policy[:, i], mode='lines', name=action, line=dict(color=colors[i])),
            row=1, col=1
        )

    # Add lines for each action for Group 2
    for i, action in enumerate(actions):
        fig.add_trace(
            go.Scatter(x=list(range(group2_policy.shape[0])), y=group2_policy[:, i], mode='lines', name=action, line=dict(color=colors[i]), showlegend=False),
            row=1, col=2
        )

    # Update layout
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="Rounds",
        yaxis_title="Average Preference",
        showlegend=True
    )

    fig.update_xaxes(title_text="Rounds", row=1, col=1)
    fig.update_xaxes(title_text="Rounds", row=1, col=2)
    fig.update_yaxes(title_text="Average Preference", row=1, col=1)
    fig.update_yaxes(title_text="Average Preference", row=1, col=2)
    if save_dir:
        safe_title = title.replace(" ", "_").replace(":", "").replace("/", "_")
        save_path = f"{save_dir}/{safe_title}.png"
        pio.write_image(fig, save_path, scale=2)  # Ensure no duplicate 'format'
        print(f"Graph saved as: {save_path}")

    # Display the figure
    fig.show()



def plot_value_comparison_by_states(V_all, group_assignment, HP, title="Value Function Comparison by States", save_dir=None):

    n_agents = HP['n_agents']
    n_rounds = int(HP['nt'] / n_agents)

    # קבוצות
    group1_agents = np.where(group_assignment == 0)[0]
    group2_agents = np.where(group_assignment == 1)[0]

    # ממוצע ערכי מצב לפי קבוצה
    avg_values_group1 = V_all[:n_rounds, group1_agents, :].mean(axis=1)  # ממוצע על סוכנים
    avg_values_group2 = V_all[:n_rounds, group2_agents, :].mean(axis=1)  # ממוצע על סוכנים

    # ציר הזמן (סיבובים)
    rounds = np.arange(n_rounds)

    # יצירת גרף
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Group 1", "Group 2"))

    # גרף עבור קבוצה 1
    fig.add_trace(go.Scatter(x=rounds, y=avg_values_group1[:, 0], mode='lines', name='State 1 (Group 1)', line=dict(color='blue')),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=rounds, y=avg_values_group1[:, 1], mode='lines', name='State 2 (Group 1)', line=dict(color='red')),
                  row=1, col=1)

    # גרף עבור קבוצה 2
    fig.add_trace(go.Scatter(x=rounds, y=avg_values_group2[:, 0], mode='lines', name='State 1 (Group 2)', line=dict(color='blue')),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=rounds, y=avg_values_group2[:, 1], mode='lines', name='State 2 (Group 2)', line=dict(color='red')),
                  row=1, col=2)

    # עדכון עיצוב הגרף
    fig.update_layout(
        title=title,
        xaxis_title="Rounds",
        yaxis_title="Value Function",
        template="plotly_white",
        legend_title="States",
        showlegend=True,
        width=1000,
        height=500
    )

    # שמירת הגרף אם יש תיקייה
    if save_dir:
        save_path = f"{save_dir}/{title.replace(' ', '_')}.png"
        fig.write_image(save_path)
        print(f"Graph saved as: {save_path}")

    fig.show()











In [ ]:
# יצירת נתונים לרשת מסורתית
adj_matrix_traditional, group_assignment_traditional = create_grouped_adj_matrix(HP_group_traditional)
reward_mat_traditional = create_reward_mat(HP_group_traditional, group_assignment_traditional)

# שמות הפעולות
actions = ["Group 1 Action", "Group 2 Action 1", "Neutral Action"]

# הרצת הסימולציה עבור רשת מסורתית
policy_all_traditional,V_all_traditional = interindividual_actor_critic(
    HP_group_traditional, group_assignment_traditional, reward_mat_traditional, thetas_init='uniform', return_values=False
)

# # חישוב ממוצע העדפות הפעולות עבור כל קבוצה
# group1_action_preferences = policy_all_traditional.mean(axis=(0, 1))[group_assignment_traditional == 0, :].mean(axis=0)
# group2_action_preferences = policy_all_traditional.mean(axis=(0, 1))[group_assignment_traditional == 1, :].mean(axis=0)

last_round_policy = policy_all_traditional[:, -1, :, :]  # הסיבוב האחרון בלבד

group1_action_preferences = last_round_policy[:, group_assignment_traditional == 0, :].mean(axis=(0, 1))
group2_action_preferences = last_round_policy[:, group_assignment_traditional == 1, :].mean(axis=(0, 1))



Trial #1 out of 1


In [ ]:
# # ציור הגרפים

# path = save_fig_path+'/two_groups'
# plot_group_action_preferences(group1_action_preferences, group2_action_preferences, actions, title="Action Preferences - Traditional Network",save_dir=path)
# # Plot evolution of preferences over rounds
# plot_action_preferences_over_rounds(policy_all_traditional, group_assignment_traditional, actions, title="Action Preferences Over Rounds - Traditional Network",save_dir=path)

# plot_value_comparison_by_states(V_all_traditional, group_assignment_traditional,HP_group_traditional, title="Value Function Comparison by States - Traditional Network",save_dir=path)

# # # ציור הגרפים
plot_group_action_preferences(group1_action_preferences, group2_action_preferences, actions, title="Action Preferences - Traditional Network")
# Plot evolution of preferences over rounds
plot_action_preferences_over_rounds(policy_all_traditional, group_assignment_traditional, actions, title="Action Preferences Over Rounds - Traditional Network")

plot_value_comparison_by_states(V_all_traditional, group_assignment_traditional,HP_group_traditional, title="Value Function Comparison by States - Traditional Network")

In [ ]:

adj_matrix_online, group_assignment_online = create_grouped_adj_matrix(HP_group_online)
reward_mat_online = create_reward_mat(HP_group_online, group_assignment_online)

# הרצת סימולציה לרשת מקוונת
policy_all_online, V_all_online = interindividual_actor_critic(
    HP_group_online, group_assignment_online, reward_mat_online,is_online=True, thetas_init='uniform', return_values=False
)

last_round_policy = policy_all_online[:, -1, :, :]  # הסיבוב האחרון בלבד

group1_action_preferences_online = last_round_policy[:, group_assignment_online == 0, :].mean(axis=(0, 1))
group2_action_preferences_online = last_round_policy[:, group_assignment_online == 1, :].mean(axis=(0, 1))





Trial #1 out of 10
Trial #2 out of 10
Trial #3 out of 10
Trial #4 out of 10
Trial #5 out of 10
Trial #6 out of 10
Trial #7 out of 10
Trial #8 out of 10
Trial #9 out of 10
Trial #10 out of 10


In [ ]:
# ציור הגרפים
plot_group_action_preferences(group1_action_preferences_online, group2_action_preferences_online, actions, title="Action Preferences - Online Network",save_dir=path)
# Plot evolution of preferences over rounds
plot_action_preferences_over_rounds(policy_all_online, group_assignment_online, actions, title="Action Preferences Over Rounds - online Network",save_dir=path)
plot_value_comparison_by_states(V_all_online, group_assignment_online,HP_group_online, title="Value Function Comparison by States - Online Network",save_dir=path)

# ציור הגרפים
# plot_group_action_preferences(group1_action_preferences_online, group2_action_preferences_online, actions, title="Action Preferences - Online Network")
# # Plot evolution of preferences over rounds
# plot_action_preferences_over_rounds(policy_all_online, group_assignment_online, actions, title="Action Preferences Over Rounds - online Network")
# plot_value_comparison_by_states(V_all_online, group_assignment_online,HP_group_online, title="Value Function Comparison by States - Online Network")


Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/two_groups/Action_Preferences_-_Online_Network.png


Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/two_groups/Action_Preferences_Over_Rounds_-_online_Network.png


Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/two_groups/Value_Function_Comparison_by_States_-_Online_Network.png


Natural Reward Critic point

In [ ]:
# שמירת מטריצת הקשרים
adj_matrix_traditional, group_assignment_traditional = create_grouped_adj_matrix(HP_group_traditional)
adj_matrix_online, group_assignment_online = create_grouped_adj_matrix(HP_group_online)
neutral_rewards = np.linspace(-3, 3, num=15)

# פונקציה לעדכון הרשתות עם אותה מטריצת קשרים
def run_experiment_with_fixed_adj(HP, neutral_rewards, adj_matrix, group_assignment, is_online):
    preferences = []
    for neutral_reward in neutral_rewards:
        print(f"Running simulation with neutral reward: {neutral_reward}")
        HP['rewards']['neutral'] = [neutral_reward]

        # יצירת מטריצת תגמולים
        reward_mat = create_reward_mat(HP, group_assignment)

        # הרצת הסימולציה עם מטריצת קשרים קבועה
        policy_all, _ = interindividual_actor_critic(
            HP, group_assignment, reward_mat, is_online=is_online, thetas_init='uniform', return_values=False
        )

        # חישוב העדפות לפעולה הנייטרלית
        group1_neutral_preference = np.mean(policy_all[:, -1, group_assignment == 0, 2])
        group2_neutral_preference = np.mean(policy_all[:, -1, group_assignment == 1, 2])
        preferences.append((group1_neutral_preference, group2_neutral_preference))
    return preferences

# הרצת ניסוי עבור רשת מסורתית עם מטריצת קשרים קבועה
results_traditional = run_experiment_with_fixed_adj(
    HP_group_traditional, neutral_rewards, adj_matrix_traditional, group_assignment_traditional, is_online=False
)

# הרצת ניסוי עבור רשת מקוונת עם מטריצת קשרים קבועה
results_online = run_experiment_with_fixed_adj(
    HP_group_online, neutral_rewards, adj_matrix_online, group_assignment_online, is_online=True
)


Running simulation with neutral reward: -3.0
Trial #1 out of 1
Running simulation with neutral reward: -2.5714285714285716
Trial #1 out of 1
Running simulation with neutral reward: -2.142857142857143
Trial #1 out of 1
Running simulation with neutral reward: -1.7142857142857144
Trial #1 out of 1
Running simulation with neutral reward: -1.2857142857142858
Trial #1 out of 1
Running simulation with neutral reward: -0.8571428571428572
Trial #1 out of 1
Running simulation with neutral reward: -0.4285714285714288
Trial #1 out of 1
Running simulation with neutral reward: 0.0
Trial #1 out of 1
Running simulation with neutral reward: 0.4285714285714284
Trial #1 out of 1
Running simulation with neutral reward: 0.8571428571428568
Trial #1 out of 1
Running simulation with neutral reward: 1.2857142857142856
Trial #1 out of 1
Running simulation with neutral reward: 1.7142857142857144
Trial #1 out of 1
Running simulation with neutral reward: 2.1428571428571423
Trial #1 out of 1
Running simulation with

In [ ]:
def plot_neutral_reward_experiment(neutral_rewards, results_traditional, results_online, save_dir=None):
    fig = go.Figure()

    # Calculate the mean of preferences for both groups in traditional and online networks
    mean_traditional = [(x[0] + x[1]) / 2 for x in results_traditional]
    mean_online = [(x[0] + x[1]) / 2 for x in results_online]

    # Add trace for traditional network mean
    fig.add_trace(go.Scatter(
        x=neutral_rewards,
        y=mean_traditional,
        mode='lines+markers',
        name='Mean - Traditional Network'
    ))

    # Add trace for online network mean
    fig.add_trace(go.Scatter(
        x=neutral_rewards,
        y=mean_online,
        mode='lines+markers',
        name='Mean - Online Network'
    ))

    # Update graph layout
    fig.update_layout(
        title="Impact of Neutral Reward on Mean Action Preference",
        xaxis_title="Neutral Reward Value",
        yaxis_title="Mean Neutral Action Preference",
        template="plotly_white",
        legend_title="Network Type"
    )

    # Save graph if save_dir is provided
    if save_dir:
        safe_title = "Neutral_Reward_Mean_Action_Preference".replace(" ", "_").replace(":", "").replace("/", "_")
        save_path = f"{save_dir}/{safe_title}.png"
        pio.write_image(fig, save_path, scale=2)
        print(f"Graph saved as: {save_path}")

    # Show the graph
    fig.show()

In [ ]:

plot_neutral_reward_experiment(neutral_rewards, results_traditional, results_online,save_dir=save_fig_path+'/two_groups')

Graph saved as: /content/drive/MyDrive/Seminarion/Graphs/two_groups/Neutral_Reward_Mean_Action_Preference.png


Polarization:


---



---



---



In [ ]:
def plot_polarization_vs_parameter(param_values, param_name, HP, update_function, save_dir=None):
    polarization_levels = []
    neutral_action_preferences = []

    for value in param_values:
        HP_copy = deepcopy(HP)
        update_function(HP_copy, value)

        # יצירת מטריצת שכנויות ושיוך קבוצתי
        adj_matrix, group_assignment = create_grouped_adj_matrix(HP_copy)
        reward_mat = create_reward_mat(HP_copy, group_assignment)

        # הרצת הסימולציה
        policy_all, _ = interindividual_actor_critic(
            HP_copy, group_assignment, reward_mat, is_online=True
        )

        # חישוב ההסתברויות לפעולות עבור קבוצה 1
        group1_action0 = np.mean(policy_all[:, -1, group_assignment == 0, 0])
        group1_action1 = np.mean(policy_all[:, -1, group_assignment == 0, 1])
        group1_neutral = np.mean(policy_all[:, -1, group_assignment == 0, 2])
        print("policy_all[:, -1, group_assignment == 0, 0]",policy_all[:, -1, group_assignment == 0, 0])
        print("policy_all[:, -1, group_assignment == 0, 1]",policy_all[:, -1, group_assignment == 0, 1])
        print("policy_all[:, -1, group_assignment == 0, 2]",policy_all[:, -1, group_assignment == 0, 2])
        # חישוב ההסתברויות לפעולות עבור קבוצה 2
        group2_action0 = np.mean(policy_all[:, -1, group_assignment == 1, 0])
        group2_action1 = np.mean(policy_all[:, -1, group_assignment == 1, 1])
        group2_neutral = np.mean(policy_all[:, -1, group_assignment == 1, 2])

        # ממוצע ההסתברות לבחור בפעולה הנייטרלית
        neutral_action_preference = (group1_neutral + group2_neutral) / 2
        neutral_action_preferences.append(neutral_action_preference)

        # חישוב הפערים בין הקבוצות
        group1_polarization = group1_action0 - group1_action1
        group2_polarization = group2_action1 - group2_action0

        # שקלול עם השפעת הפעולה הנייטרלית
        polarization = ((group1_polarization + group2_polarization) / 2) * (1 - neutral_action_preference)
        polarization_levels.append(polarization)

    # חישוב מתאם פירסון
    correlation, _ = pearsonr(param_values, polarization_levels)

    # ציור הגרף
    fig = go.Figure()

    # קו הפולריזציה
    fig.add_trace(go.Scatter(
        x=param_values,
        y=polarization_levels,
        mode='markers+lines',
        name='Polarization Level',
        marker=dict(size=10, color='blue'),
        line=dict(dash='solid')
    ))

    # קו הפעולה הנייטרלית
    fig.add_trace(go.Scatter(
        x=param_values,
        y=neutral_action_preferences,
        mode='lines',
        name='Neutral Action Preference',
        line=dict(color='green', width=2, dash='solid'),
        opacity=0.8
    ))

    # קו השיפוע הממוצע
    trendline = np.poly1d(np.polyfit(param_values, polarization_levels, 1))
    fig.add_trace(go.Scatter(
        x=param_values,
        y=trendline(param_values),
        mode='lines',
        name='Trendline (Pearson)',
        line=dict(color='red', dash='dash')
    ))

    # עיצוב הגרף
    fig.update_layout(
        title=f"Effect of {param_name} on Polarization Online SN(\u03c1 = {correlation:.2f})",
        xaxis_title=param_name,
        yaxis_title="Weighted Polarization Level",
        template="plotly_white"
    )

    if save_dir:
        safe_title = param_name.replace(" ", "_").replace(":", "").replace("/", "_")
        save_path = f"{save_dir}/{safe_title}_Online_Polarization.png"
        fig.write_image(save_path)
        print(f"Graph saved at: {save_path}")

    fig.show()


In [ ]:
ratios = [0.01,0.05,0.1,0.2,0.4,0.5,0.6,0.75,0.9,1]
connection_strengths = np.linspace(0, 1, 10)
network_sizes = [5,10,50,100,150,200]
connection_learning_rates = np.logspace(-4, -1, 10)



def update_ratio(HP, value):
    HP['ratio'] = value

def update_connection_strength(HP, value):
    HP['adj_mat_group_conn']['in_group'] = value
    HP['adj_mat_group_conn']['out_group'] = value / 2  # לדוגמה, קשרים מחוץ לקבוצה חלשים יותר


def update_network_size(HP, value):
    HP['n_agents'] = int(value)
    HP['nt'] = int(value * 20)  # התאמת מספר הצעדים בהתאם למספר הסוכנים

def update_connection_learning_rate(HP, value):
    HP['lrm'] = value


traditional:

In [ ]:
pol_path = save_fig_path+'/two_groups/polarization_corr'

plot_polarization_vs_parameter(ratios, "Ratios", HP_group_traditional, update_ratio,save_dir=pol_path)

plot_polarization_vs_parameter(connection_strengths, "Connection Strength", HP_group_traditional, update_connection_strength,save_dir=pol_path)

plot_polarization_vs_parameter(network_sizes, "SN size", HP_group_traditional, update_network_size,save_dir=pol_path)

plot_polarization_vs_parameter(connection_learning_rates, "Dynamic", HP_group_traditional, update_connection_learning_rate,save_dir=pol_path)


Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.29616028 0.29313222 0.28931131 0.28973772 0.29328561]]
policy_all[:, -1, group_assignment == 0, 1] [[0.2918186  0.28843481 0.27139644 0.28777364 0.27851158]]
policy_all[:, -1, group_assignment == 0, 2] [[0.41202112 0.41843297 0.43929225 0.42248864 0.42820281]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.3073841  0.34116695 0.32787365 0.32839722 0.31173038]]
policy_all[:, -1, group_assignment == 0, 1] [[0.25692293 0.25815473 0.26198433 0.25304886 0.25701416]]
policy_all[:, -1, group_assignment == 0, 2] [[0.43569297 0.40067832 0.41014202 0.41855392 0.43125546]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.36399092 0.37678633 0.36832918 0.32312049 0.35068312]]
policy_all[:, -1, group_assignment == 0, 1] [[0.20926566 0.22518915 0.21080182 0.20216014 0.22707393]]
policy_all[:, -1, group_assignment == 0, 2] [[0.42674342 0.39802452 0.42086899 0.47471937 0.42224295]]
Trial #1 out of 1
p

Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.59088388 0.61152243 0.50940177 0.58148777 0.56947213]]
policy_all[:, -1, group_assignment == 0, 1] [[0.12583909 0.10401627 0.15450065 0.16133457 0.1440213 ]]
policy_all[:, -1, group_assignment == 0, 2] [[0.28327703 0.2844613  0.33609758 0.25717766 0.28650657]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.67159483 0.72026764 0.47013612 0.63073647 0.6221012 ]]
policy_all[:, -1, group_assignment == 0, 1] [[0.08574398 0.08586371 0.15517778 0.120984   0.10625342]]
policy_all[:, -1, group_assignment == 0, 2] [[0.24266119 0.19386865 0.3746861  0.24827953 0.27164538]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.61409161 0.59395305 0.55975931 0.53459813 0.72196987]]
policy_all[:, -1, group_assignment == 0, 1] [[0.10919317 0.13184745 0.13724443 0.15266137 0.05715954]]
policy_all[:, -1, group_assignment == 0, 2] [[0.27671522 0.2741995  0.30299626 0.31274049 0.2208706 ]]
Trial #1 out of 1
p

Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.40855783 0.46745313 0.46148322]]
policy_all[:, -1, group_assignment == 0, 1] [[0.25109319 0.19857235 0.22206478]]
policy_all[:, -1, group_assignment == 0, 2] [[0.34034898 0.33397453 0.316452  ]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.54428939 0.53947624 0.59684457 0.50796137 0.5337259 ]]
policy_all[:, -1, group_assignment == 0, 1] [[0.16630009 0.1955857  0.15149214 0.16494009 0.18164014]]
policy_all[:, -1, group_assignment == 0, 2] [[0.28941052 0.26493806 0.25166329 0.32709854 0.28463395]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.86020124 0.88420441 0.87821952 0.84632528 0.8559369  0.89225202
  0.86445921 0.87323837 0.88247298 0.91061011 0.90113941 0.87529522
  0.89744097 0.90498532 0.86186541 0.93201064 0.83167187 0.87916704
  0.91808523 0.90396278 0.90200618 0.90622126 0.91352178 0.91640293
  0.88806106]]
policy_all[:, -1, group_assignment == 0, 1] [[0.03682986 0.0550

Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.7531144  0.7407653  0.56137879 0.79903056 0.76827179]]
policy_all[:, -1, group_assignment == 0, 1] [[0.10772604 0.10975567 0.09135559 0.05372023 0.10359756]]
policy_all[:, -1, group_assignment == 0, 2] [[0.13915956 0.14947903 0.34726562 0.1472492  0.12813065]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.77435449 0.75068003 0.77270771 0.5203351  0.67534826]]
policy_all[:, -1, group_assignment == 0, 1] [[0.08436409 0.0694538  0.0707969  0.07103064 0.07969461]]
policy_all[:, -1, group_assignment == 0, 2] [[0.14128142 0.17986617 0.15649539 0.40863426 0.24495713]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.7423164  0.71128383 0.73514577 0.7242109  0.70796379]]
policy_all[:, -1, group_assignment == 0, 1] [[0.06770884 0.09976291 0.06772944 0.07662103 0.08280813]]
policy_all[:, -1, group_assignment == 0, 2] [[0.18997477 0.18895326 0.19712479 0.19916807 0.20922808]]
Trial #1 out of 1
p

Online:

In [ ]:
pol_path = save_fig_path+'/two_groups/polarization_corr'

plot_polarization_vs_parameter(network_sizes, "SN size", HP_group_online, update_network_size,save_dir=pol_path)

plot_polarization_vs_parameter(ratios, "Ratios", HP_group_online, update_ratio,save_dir=pol_path)

plot_polarization_vs_parameter(connection_strengths, "Connection Strength", HP_group_online, update_connection_strength,save_dir=pol_path)


plot_polarization_vs_parameter(connection_learning_rates, "Dynamic", HP_group_online, update_connection_learning_rate,save_dir=pol_path)

Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.32064284 0.34773452 0.32822877]]
policy_all[:, -1, group_assignment == 0, 1] [[0.30749216 0.30394704 0.30513708]]
policy_all[:, -1, group_assignment == 0, 2] [[0.371865   0.34831844 0.36663414]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.35129919 0.35187753 0.3556233  0.34583066 0.36858607]]
policy_all[:, -1, group_assignment == 0, 1] [[0.27743229 0.27888891 0.29209741 0.2965362  0.28839989]]
policy_all[:, -1, group_assignment == 0, 2] [[0.37126852 0.36923356 0.35227929 0.35763314 0.34301403]]
Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.52175382 0.46870547 0.5041759  0.55054601 0.4055106  0.47100897
  0.49444232 0.46869431 0.46694688 0.47981537 0.53408584 0.43811582
  0.48557651 0.5117053  0.4828734  0.51253355 0.49270021 0.46969388
  0.52252891 0.36950646 0.53162495 0.43158034 0.4275695  0.49646862
  0.52904254]]
policy_all[:, -1, group_assignment == 0, 1] [[0.19710753 0.1925

Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.35274686 0.36377323 0.35307379 0.33341144 0.33338822 0.34620125
  0.33601301 0.31826752 0.34797557 0.38054786 0.33998051 0.31026848
  0.33136504 0.37620792 0.33266351 0.36516966 0.34496923 0.3133254
  0.35371227 0.37425931 0.37595492 0.41102011 0.34004418 0.32084291
  0.33362355 0.33848739 0.33439741 0.35009912 0.37887584 0.35440718
  0.41163654 0.36201445 0.36274084 0.36922257 0.34253342 0.3302262
  0.34959888 0.36039676 0.33401394 0.3460258  0.33816165 0.39637775
  0.346858   0.35570118 0.36280892 0.3384156  0.35533509 0.32795307
  0.33290402 0.39333666]]
policy_all[:, -1, group_assignment == 0, 1] [[0.21746584 0.20729375 0.20223601 0.21470994 0.21812733 0.22161701
  0.21974588 0.23996216 0.22498055 0.22443967 0.21334452 0.21600273
  0.2470725  0.22266248 0.21041885 0.20022762 0.22504108 0.20060048
  0.22583908 0.2098276  0.21134536 0.21360674 0.19827761 0.21964561
  0.22646934 0.21537297 0.2222697  0.20792034 0.201123

Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.76517719 0.79507435 0.77730774 0.78600235 0.75698919 0.79686966
  0.81067032 0.78959664 0.79007411 0.77243037 0.72982252 0.78506368
  0.82043097 0.80191242 0.73573291 0.77481474 0.75168848 0.78715737
  0.77575334 0.8159349  0.81705329 0.71518514 0.6528008  0.81865863
  0.72240421 0.74300204 0.7511572  0.76813372 0.79438356 0.82678819
  0.73997509 0.78190622 0.83085175 0.72525128 0.80395417 0.75826348
  0.81162822 0.71865469 0.77348306 0.73585431 0.83239678 0.79621679
  0.82064287 0.81941097 0.82383374 0.76158698 0.77409991 0.81144957
  0.75629191 0.75064884]]
policy_all[:, -1, group_assignment == 0, 1] [[0.09730849 0.04741955 0.064335   0.07300181 0.10104378 0.08113717
  0.06805482 0.06991946 0.06827735 0.08294327 0.10823414 0.07190395
  0.05058191 0.07195986 0.0962715  0.07823412 0.07478161 0.06573205
  0.08081489 0.04690126 0.06149573 0.08488621 0.12554279 0.0636512
  0.07304727 0.05528471 0.08341233 0.10657095 0.06060

Trial #1 out of 1
policy_all[:, -1, group_assignment == 0, 0] [[0.7886461  0.87167123 0.8772979  0.81424587 0.84231396 0.78827517
  0.80574453 0.87307394 0.84778086 0.86602492 0.81958205 0.86099059
  0.7869904  0.86055908 0.80823873 0.8383231  0.85735104 0.86931288
  0.82645676 0.830697   0.83498883 0.81862122 0.81854519 0.86459792
  0.83142731 0.81389023 0.77147266 0.79111001 0.83999954 0.83282516
  0.86192847 0.81392376 0.78773095 0.84057021 0.83624147 0.80035932
  0.81520807 0.79165483 0.86439028 0.77096969 0.85151918 0.81267642
  0.8635359  0.86038916 0.84902005 0.80897218 0.84453384 0.78869957
  0.80920187 0.84998508]]
policy_all[:, -1, group_assignment == 0, 1] [[0.06615114 0.04631707 0.03558763 0.04700785 0.04318051 0.07711942
  0.05828235 0.03363421 0.05301917 0.03922467 0.05590212 0.03585632
  0.0646357  0.03812495 0.05783151 0.03950966 0.03890737 0.02798805
  0.0580375  0.04571461 0.0471968  0.04891522 0.04508469 0.03320811
  0.03335751 0.06953198 0.03681417 0.06502478 0.0427